# CAsT 2020: Query Rewriting với Ollama Cloud + BM25

Notebook này chạy trên Colab và gọi **Ollama Cloud** (`https://ollama.com`), không dùng Ollama local.

Trước khi chạy, tạo API key tại [ollama.com/settings/keys](https://ollama.com/settings/keys) rồi thêm Colab Secret tên `OLLAMA_API_KEY`.

Ba loại truy vấn trên cùng BM25:

- **Raw**: câu hỏi hiện tại.
- **LLM**: câu hỏi được Ollama Cloud viết lại dựa trên lịch sử hội thoại.
- **Human**: bản viết lại thủ công của CAsT 2020.

Luồng: tải dữ liệu → mở index → Raw/Human → Ollama Cloud → LLM retrieval → so sánh.


In [4]:
# Cài các thư viện cần dùng. Colab cần Java 21 cho Pyserini.
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-21-jdk-headless"], check=True)

# Cài đặt Pillow 10.4.0 trước để đảm bảo tính tương thích trên Python 3.13
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "Pillow==10.4.0"
], check=True)

# Cài đặt pyserini==0.22.1 để tương thích với định dạng index cast2019 (Lucene 8.x)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyserini==0.22.1", "ir-datasets==0.6.3", "ir-measures==0.4.3",
    "pandas", "tqdm", "ollama"
], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'pyserini==0.22.1', 'ir-datasets==0.6.3', 'ir-measures==0.4.3', 'pandas', 'tqdm', 'ollama'], returncode=0)

In [2]:
# Tự động cài đặt faiss-cpu nếu chưa có để sửa lỗi ModuleNotFoundError: No module named 'faiss'
import sys, subprocess
try:
    import faiss
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=True)

# Cấu hình: Ollama Cloud trên Colab, không dùng localhost:11434.
# Đã đồng bộ cấu hình tương thích cho Pillow trên Python 3.13.
from pathlib import Path
import json
import os

import ir_datasets
import ir_measures
import pandas as pd
from ir_measures import nDCG, Recall, RR
from ollama import Client
from pyserini.search.lucene import LuceneSearcher
from tqdm.auto import tqdm

MODEL = "qwen3.5:9b"  # tên model cloud; đổi nếu muốn, xem https://ollama.com/search?c=cloud
OLLAMA_HOST = "https://ollama.com"
TOP_K = 100
K1, B = 0.9, 0.4

OUTPUT = Path("cast2020_results")
OUTPUT.mkdir(exist_ok=True)

from google.colab import userdata
api_key = userdata.get("OLLAMA_API_KEY")
if not api_key:
    raise RuntimeError("Chưa cấu hình Colab Secret OLLAMA_API_KEY")

client = Client(
    host=OLLAMA_HOST,
    headers={"Authorization": f"Bearer {api_key}"},
)
del api_key

print({"model": MODEL, "ollama_host": OLLAMA_HOST, "top_k": TOP_K})


{'model': 'qwen3.5:9b', 'ollama_host': 'https://ollama.com', 'top_k': 100}


In [3]:
# Đọc 216 lượt để tạo history; chỉ 208 lượt có qrels được dùng để chấm điểm.
full = ir_datasets.load("trec-cast/v1/2020")
judged_dataset = ir_datasets.load("trec-cast/v1/2020/judged")

qrels = pd.DataFrame([
    {"query_id": q.query_id, "doc_id": q.doc_id, "relevance": q.relevance}
    for q in judged_dataset.qrels_iter()
])
judged_ids = set(qrels.query_id)

history_by_topic = {}
turns = []
for q in sorted(full.queries_iter(), key=lambda x: (x.topic_number, x.turn_number)):
    history = history_by_topic.setdefault(q.topic_number, [])
    turns.append({
        "query_id": q.query_id,
        "topic": q.topic_number,
        "turn": q.turn_number,
        "raw": q.raw_utterance,
        "human": q.manual_rewritten_utterance,
        "history": history.copy(),
    })
    history.append(q.raw_utterance)

turns = pd.DataFrame(turns)
judged = turns[turns.query_id.isin(judged_ids)].reset_index(drop=True)

assert len(turns) == 216
assert len(judged) == 208
assert set(judged.query_id) == judged_ids

print(f"{len(turns)} total turns, {len(judged)} judged turns, {len(qrels):,} qrels")
judged.head(3)

216 total turns, 208 judged turns, 40,451 qrels


,query_id,topic,turn,raw,human,history
0,81_1,81,1,How do you know when your garage door opener i...,How do you know when your garage door opener i...,[]
1,81_2,81,2,Now it stopped working. Why?,Now my garage door opener stopped working. Why?,[How do you know when your garage door opener ...
2,81_3,81,3,How much does it cost for someone to fix it?,How much does it cost for someone to repair a ...,[How do you know when your garage door opener ...


In [4]:
# Pyserini tự tải, kiểm tra và giải nén prebuilt index cast2019.
searcher = LuceneSearcher.from_prebuilt_index("cast2019", verbose=True)
searcher.set_bm25(K1, B)

assert searcher.num_docs == 38_429_835

# Cổng an toàn: mọi document ID trong qrels phải tồn tại trong index.
missing = [
    doc_id
    for doc_id in tqdm(qrels.doc_id.unique(), desc="Checking qrel document IDs")
    if searcher.doc(doc_id) is None
]
assert not missing, f"Qrels và index không tương thích; ví dụ: {missing[:5]}"

smoke_hits = searcher.search(judged.iloc[0].raw, k=3)
assert smoke_hits
[(hit.docid, hit.score) for hit in smoke_hits]

Attempting to initialize pre-built index cast2019.
/root/.cache/pyserini/indexes/index-cast2019.36e604d7f5a4e08ade54e446be2f6345 already exists, skipping download.
{'total_terms': 1593628213, 'documents': 38429835, 'non_empty_documents': 38426205, 'unique_terms': -1}
Index passes consistency checks against pre-built index 'cast2019'!
Initializing cast2019...


Checking qrel document IDs:   0%|          | 0/29264 [00:00<?, ?it/s]

[('MARCO_6154874', 23.265274047851562),
 ('MARCO_6568085', 21.818201065063477),
 ('MARCO_8454278', 21.524202346801758)]

In [5]:
# Hai hàm nhỏ dùng chung cho cả Raw, LLM và Human.
MEASURES = [nDCG @ 10, Recall @ 100, RR(rel=1) @ 10]

def retrieve(queries, name):
    rows = []
    for query_id, text in tqdm(queries.items(), desc=f"Retrieving {name}"):
        for rank, hit in enumerate(searcher.search(text, k=TOP_K), start=1):
            rows.append({
                "query_id": query_id,
                "doc_id": hit.docid,
                "rank": rank,
                "score": hit.score,
                "method": name,
            })
    return pd.DataFrame(rows)

def evaluate(run):
    aggregate = {
        str(measure): score
        for measure, score in ir_measures.calc_aggregate(MEASURES, qrels, run).items()
    }
    per_query = pd.DataFrame([
        {"query_id": result.query_id, "metric": str(result.measure), "value": result.value}
        for result in ir_measures.iter_calc(MEASURES, qrels, run)
    ]).pivot(index="query_id", columns="metric", values="value")
    return aggregate, per_query.reindex(judged.query_id).fillna(0)


In [6]:
# Baselines: Raw và Human chạy trước, không liên quan tới Ollama.
query_table = judged.set_index("query_id")
runs, scores, per_query = {}, {}, {}

for method, column in {"raw": "raw", "human": "human"}.items():
    runs[method] = retrieve(query_table[column], method)
    scores[method], per_query[method] = evaluate(runs[method])
    runs[method].to_csv(OUTPUT / f"{method}_run.csv", index=False)

pd.DataFrame(scores).T

Retrieving raw: 0it [00:00, ?it/s]

Retrieving human: 0it [00:00, ?it/s]

,nDCG@10,RR@10,R@100
raw,0.077960,0.172014,0.125089
human,0.255781,0.542960,0.427310


In [8]:
# Ollama Cloud viết lại từng query. Chạy lại cell này sẽ tiếp tục từ file cache hiện có.
SYSTEM_PROMPT = """Rewrite the current conversational search utterance as one standalone search query.
Use only facts from the preceding user utterances. Preserve intent, entities, numbers,
dates, constraints, negation, and language. Do not answer the question or explain.
Return only the rewritten query on one line."""

CACHE = OUTPUT / "ollama_rewrites.csv"
cached = pd.read_csv(CACHE).to_dict("records") if CACHE.exists() else []
done = {row["query_id"] for row in cached}

for turn in tqdm(judged.itertuples(index=False), total=len(judged), desc="Ollama Cloud rewriting"):
    if turn.query_id in done:
        continue

    user_input = json.dumps({
        "history": turn.history,
        "current_query": turn.raw,
    }, ensure_ascii=False)

    response = client.chat(
        model="gemma4:31b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_input},
        ],
        think=False,
        options={"temperature": 0, "num_predict": 128},
    )

    rewrite = (response.message.content or "").strip()
    if not rewrite or "\n" in rewrite:
        raise ValueError(f"Output không hợp lệ tại {turn.query_id}: {rewrite!r}")

    cached.append({"query_id": turn.query_id, "rewrite": rewrite, "model": MODEL})
    pd.DataFrame(cached).to_csv(CACHE, index=False)
    done.add(turn.query_id)

rewrites = pd.DataFrame(cached)
assert set(rewrites.query_id) == judged_ids
rewrites.head()

Ollama Cloud rewriting:   0%|          | 0/208 [00:00<?, ?it/s]

,query_id,rewrite,model
0,81_1,signs your garage door opener is failing,qwen3.5:9b
1,81_2,Why would a garage door opener stop working?,qwen3.5:9b
2,81_3,How much does it cost to have a broken garage ...,qwen3.5:9b
3,81_4,How much does it cost to replace a garage door...,qwen3.5:9b
4,81_5,How do I choose a new garage door opener?,qwen3.5:9b


In [9]:
# Retrieval cho LLM và bảng kết quả cuối cùng.
llm_queries = rewrites.set_index("query_id").rewrite.reindex(judged.query_id)
runs["llm"] = retrieve(llm_queries, "llm")
scores["llm"], per_query["llm"] = evaluate(runs["llm"])
runs["llm"].to_csv(OUTPUT / "llm_run.csv", index=False)

summary = pd.DataFrame(scores).T.loc[["raw", "llm", "human"]]
summary.to_csv(OUTPUT / "aggregate_metrics.csv")
display(summary)

# Delta theo từng query: số dương nghĩa là LLM tốt hơn Raw.
delta = per_query["llm"] - per_query["raw"]
delta["classification"] = delta["nDCG@10"].map(
    lambda x: "improved" if x > 0 else "degraded" if x < 0 else "tied"
)
delta = judged[["query_id", "topic", "turn", "raw", "human"]].merge(
    rewrites[["query_id", "rewrite"]], on="query_id"
).merge(delta.reset_index(), on="query_id")
delta.to_csv(OUTPUT / "llm_minus_raw_per_query.csv", index=False)

delta.sort_values("nDCG@10", ascending=False).head(10)

Retrieving llm: 0it [00:00, ?it/s]

,nDCG@10,RR@10,R@100
raw,0.077960,0.172014,0.125089
llm,0.231603,0.515753,0.369856
human,0.255781,0.542960,0.427310


,query_id,topic,turn,raw,human,rewrite,R@100,RR@10,nDCG@10,classification
69,89_4,89,4,Where is it native to?,Where is the Venus flytrap native to?,Where are Venus flytraps native to?,0.928571,1.0,0.736447,improved
159,100_7,100,7,I meant medicare,What is the coverage of the crown in medicare?,What is the Medicare coverage for a dental crown?,0.650000,1.0,0.663864,improved
52,87_5,87,5,Tell me about the Hamlin variety.,Tell me about the Hamlin orange variety.,Hamlin orange tree variety details,0.727273,1.0,0.642967,improved
73,89_8,89,8,Why would the roles reverse?,Why would the roles of predator and prey reverse?,Why would the roles of predator and prey reverse?,0.692308,1.0,0.639212,improved
60,88_5,88,5,What was the culture like?,What was the culture like in the Ottoman Empire?,What was the culture of slavery in the labor s...,0.351351,1.0,0.615865,improved
17,82_10,82,10,How could Co-Extra improve it?,How could Co-Extra improve DNA-based testing f...,How could Co-Extra improve DNA-based testing f...,0.285714,1.0,0.614278,improved
12,82_5,82,5,Tell me more about traceability tools.,Tell me about traceability tools for GMO foods...,traceability tools for GMO food labeling in th...,0.375000,1.0,0.585601,improved
6,81_7,81,7,What's important for me to know about their sa...,What's important for me to know about the safe...,What are the important safety considerations f...,0.600000,1.0,0.553146,improved
57,88_2,88,2,What was the role of slavery?,What was the role of slavery in the labor syst...,What was the role of slavery in the labor syst...,0.463415,1.0,0.551636,improved
121,95_8,95,8,How is corn oil used?,How is corn oil used to make biodegradable pla...,How is corn oil used in the production of biod...,0.468750,1.0,0.523921,improved


In [10]:
import shutil

# Nén thư mục kết quả cast2020_results thành file zip
output_filename = "cast2020_results"
shutil.make_archive(output_filename, 'zip', OUTPUT)

print(f"Đã tạo thành công file: {output_filename}.zip")

Đã tạo thành công file: cast2020_results.zip
